# v2 Baseline Training

Train a single RL jockey (PPO) against scripted opponents using the v2 simulation port.

**v2 changes from v1:**
- Discrete 5×5 action space (25 actions) instead of continuous
- 139-float observation vector (14 self + 10 track + 115 opponents)
- Drain-only stamina model with binary exhaustion
- SAT-based OBB collision
- 240Hz / 8-substep physics

**Reward:** Delta-progress + finish bonus + exhaustion penalty (−0.001/tick at 0 stamina).

**Auto-resume:** Checkpoints and exported ONNX files save to Google Drive. If the runtime disconnects, re-run all cells — training picks up from the latest checkpoint.

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION — edit these before running
# ============================================================

# Training version — labels the checkpoint folder on Drive.
# Bump this when changing reward, physics, or action mapping
# so previous runs are preserved.
VERSION = "v2.0"

# Track to train on
TRACK = "tracks/test_oval.json"

# Number of horses (agent + scripted opponents)
HORSE_COUNT = 4

# Total training timesteps
TOTAL_TIMESTEPS = 1_000_000

# Max steps per episode before truncation
MAX_STEPS = 5000

# Parallel envs (Colab typically has 2 CPUs, use 2-4)
N_ENVS = 4

# Checkpoint dir on Google Drive
DRIVE_BASE = "/content/drive/MyDrive/hr-checkpoints"
DRIVE_DIR = f"{DRIVE_BASE}/{VERSION}"

# Resume: 'auto' finds latest .zip, False = start fresh
RESUME = 'auto'

# Logging
LOG_EVERY = 10          # print every N episodes
SAVE_EVERY = 50_000     # checkpoint every N timesteps

## Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update repo
import os

REPO_DIR = "/content/hr-simulation"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Install uv for fast dependency resolution
!pip install -q uv

# Install project deps into Colab's system Python (avoids .venv C extension mismatch)
!uv pip install --system -e "."

## Resolve Checkpoint

In [ ]:
import glob
from pathlib import Path

os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Version: {VERSION}")
print(f"Track: {TRACK}")
print(f"Horses: {HORSE_COUNT}")
print(f"Timesteps: {TOTAL_TIMESTEPS:,}")
print(f"Checkpoint dir: {DRIVE_DIR}")
print()

# Find checkpoint to resume from
restore_path = None
restored_timesteps = 0

if RESUME is False:
    print("Starting fresh (RESUME=False)")
elif RESUME == 'auto':
    # Look for latest checkpoint in Drive dir
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        restore_path = zips[-1]
        # Extract timestep count from filename: checkpoint_<N>_steps.zip
        stem = Path(restore_path).stem
        parts = stem.split("_")
        for i, p in enumerate(parts):
            if p == "steps" and i > 0:
                try:
                    restored_timesteps = int(parts[i - 1])
                except ValueError:
                    pass
        print(f"Auto-resume: {restore_path}")
        print(f"  Restored timesteps: {restored_timesteps:,}")
    else:
        # Check for final model
        final = os.path.join(DRIVE_DIR, "final_model.zip")
        if os.path.exists(final):
            restore_path = final
            restored_timesteps = TOTAL_TIMESTEPS
            print(f"Found final model: {final}")
            print("Training already complete!")
        else:
            print("No checkpoint found \u2014 starting fresh")
elif isinstance(RESUME, str):
    restore_path = RESUME
    print(f"Resuming from: {restore_path}")
else:
    print("Starting fresh")

remaining_timesteps = max(0, TOTAL_TIMESTEPS - restored_timesteps)
print(f"\nRemaining timesteps: {remaining_timesteps:,}")

## Training

In [ ]:
import time
import json
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv

from horse_racing.env import HorseRacingSingleEnv


class ColabLoggingCallback(BaseCallback):
    """Logs episode stats and saves history to Drive."""

    def __init__(self, log_every=10, total_timesteps=1_000_000, history_path=None):
        super().__init__(verbose=0)
        self._log_every = log_every
        self._total_timesteps = total_timesteps
        self._history_path = history_path
        self._ep_count = 0
        self._start = time.time()
        self.history = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self._ep_count += 1
                ep = info["episode"]
                self.history.append({
                    "episode": self._ep_count,
                    "timesteps": self.num_timesteps,
                    "reward": float(ep["r"]),
                    "length": int(ep["l"]),
                })
                if self._ep_count % self._log_every == 0:
                    elapsed = time.time() - self._start
                    ts = self.num_timesteps
                    rate = ts / elapsed if elapsed > 0 else 0
                    pct = ts / self._total_timesteps * 100
                    eta_s = (self._total_timesteps - ts) / rate if rate > 0 else 0
                    eta_m = eta_s / 60
                    recent = self.history[-10:]
                    avg_r = np.mean([h["reward"] for h in recent])
                    avg_l = np.mean([h["length"] for h in recent])
                    print(
                        f"  [{pct:5.1f}%] ep {self._ep_count:5d} | "
                        f"reward: {avg_r:8.3f} | "
                        f"len: {avg_l:6.0f} | "
                        f"ts: {ts:>10,} | "
                        f"{rate:,.0f} ts/s | "
                        f"ETA: {eta_m:.0f}m"
                    )
                # Save history periodically
                if self._history_path and self._ep_count % (self._log_every * 5) == 0:
                    with open(self._history_path, "w") as f:
                        json.dump(self.history, f)
        return True


def make_env(track_path, horse_count, max_steps):
    def _init():
        env = HorseRacingSingleEnv(
            track_path=track_path,
            horse_count=horse_count,
            max_steps=max_steps,
        )
        return Monitor(env)
    return _init

In [ ]:
if remaining_timesteps <= 0:
    print("Training already complete \u2014 skip to evaluation or export.")
else:
    vec_env = SubprocVecEnv([make_env(TRACK, HORSE_COUNT, MAX_STEPS) for _ in range(N_ENVS)])

    policy_kwargs = dict(net_arch=[256, 256])

    if restore_path:
        print(f"Loading model from {restore_path}")
        model = PPO.load(restore_path, env=vec_env, device="cpu")
        # Lower LR for resumed training to avoid destabilizing
        finetune_lr = 1e-4
        model.learning_rate = finetune_lr
        model._setup_lr_schedule()
        print(f"  Fine-tune LR: {finetune_lr}")
    else:
        model = PPO(
            "MlpPolicy",
            vec_env,
            learning_rate=3e-4,
            gamma=0.998,
            gae_lambda=0.95,
            n_steps=2048,
            batch_size=512,
            n_epochs=10,
            clip_range=0.2,
            ent_coef=0.02,
            policy_kwargs=policy_kwargs,
            verbose=0,
            device="cpu",
        )

    print(f"Model params: {sum(p.numel() for p in model.policy.parameters()):,}")
    print()

In [ ]:
if remaining_timesteps <= 0:
    print("Skipping training \u2014 already complete.")
else:
    history_path = os.path.join(DRIVE_DIR, "history.json")

    checkpoint_cb = CheckpointCallback(
        save_freq=max(SAVE_EVERY // N_ENVS, 1),
        save_path=DRIVE_DIR,
        name_prefix="checkpoint",
    )
    logging_cb = ColabLoggingCallback(
        log_every=LOG_EVERY,
        total_timesteps=remaining_timesteps,
        history_path=history_path,
    )

    print(f"Training v2 baseline: {remaining_timesteps:,} timesteps")
    print(f"Track: {TRACK} | Horses: {HORSE_COUNT} | Envs: {N_ENVS}")
    print(f"Checkpoints \u2192 {DRIVE_DIR}")
    print()

    start_time = time.time()

    try:
        model.learn(
            total_timesteps=remaining_timesteps,
            callback=[checkpoint_cb, logging_cb],
            reset_num_timesteps=False if restore_path else True,
        )
    except KeyboardInterrupt:
        print("\nTraining interrupted.")

    # Save final model
    final_path = os.path.join(DRIVE_DIR, "final_model")
    model.save(final_path)

    # Save history
    with open(history_path, "w") as f:
        json.dump(logging_cb.history, f)

    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"Training complete")
    print(f"  Time: {elapsed/60:.1f} min")
    print(f"  Episodes: {logging_cb._ep_count}")
    print(f"  Final model: {final_path}.zip")
    print(f"{'='*60}")

    vec_env.close()

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

# Load history (from current run or saved on Drive)
history_path = os.path.join(DRIVE_DIR, "history.json")
if 'logging_cb' in dir() and logging_cb.history:
    history = logging_cb.history
elif os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
else:
    history = []
    print("No history found.")

if history:
    rewards = [h["reward"] for h in history]
    lengths = [h["length"] for h in history]
    timesteps = [h["timesteps"] for h in history]

    window = min(50, len(rewards) // 4) or 1
    smooth_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smooth_l = np.convolve(lengths, np.ones(window)/window, mode='valid')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(rewards, alpha=0.2, color='blue')
    ax1.plot(range(window-1, len(rewards)), smooth_r, color='blue', linewidth=2)
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    ax1.set_title('Episode Reward')
    ax1.grid(True, alpha=0.3)

    ax2.plot(lengths, alpha=0.2, color='green')
    ax2.plot(range(window-1, len(lengths)), smooth_l, color='green', linewidth=2)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Length (ticks)')
    ax2.set_title('Episode Length')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\nTotal episodes: {len(rewards)}")
    print(f"Last 10 avg reward: {np.mean(rewards[-10:]):.3f}")
    print(f"Last 10 avg length: {np.mean(lengths[-10:]):.0f}")

## Quick Evaluation

In [ ]:
from horse_racing.env import HorseRacingSingleEnv
from horse_racing.action import decode_action

# Load best model
final_path = os.path.join(DRIVE_DIR, "final_model.zip")
if not os.path.exists(final_path):
    # Try latest checkpoint
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        final_path = zips[-1]
    else:
        raise FileNotFoundError("No model found to evaluate")

eval_model = PPO.load(final_path, device="cpu")
print(f"Evaluating: {final_path}")

eval_episodes = 5
env = HorseRacingSingleEnv(track_path=TRACK, horse_count=HORSE_COUNT, max_steps=MAX_STEPS)

results = []
for ep in range(eval_episodes):
    obs, info = env.reset()
    total_reward = 0.0
    steps = 0
    while True:
        action, _ = eval_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))
        total_reward += reward
        steps += 1
        if terminated or truncated:
            break
    tang, norm = decode_action(int(action))
    results.append({
        "episode": ep + 1,
        "reward": total_reward,
        "steps": steps,
        "progress": info["progress"],
        "stamina": info["stamina"],
        "finish_order": info["finish_order"],
        "finished": terminated,
    })
    status = f"#{info['finish_order']}" if info['finish_order'] else ("DNF" if not terminated else "?")
    print(
        f"  ep {ep+1}: reward={total_reward:8.3f} | "
        f"steps={steps:5d} | progress={info['progress']:.3f} | "
        f"stamina={info['stamina']:.1f} | {status}"
    )

print(f"\nAvg reward: {np.mean([r['reward'] for r in results]):.3f}")
print(f"Avg steps: {np.mean([r['steps'] for r in results]):.0f}")
finishers = [r for r in results if r['finished']]
print(f"Finish rate: {len(finishers)}/{eval_episodes}")
if finishers:
    print(f"Avg finish order: {np.mean([r['finish_order'] for r in finishers]):.1f}")

## Export to ONNX

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
from horse_racing.core.observation import OBS_SIZE
from horse_racing.action import NUM_ACTIONS


class PolicyWrapper(nn.Module):
    """Wraps SB3 PPO actor for ONNX export: obs -> logits.

    v2 uses discrete actions, so the output is raw logits over 25 actions.
    The TS OnnxJockey takes argmax of these logits.
    """
    def __init__(self, policy):
        super().__init__()
        self.features_extractor = policy.features_extractor
        self.mlp_extractor = policy.mlp_extractor
        self.action_net = policy.action_net

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


# Load model
final_path = os.path.join(DRIVE_DIR, "final_model.zip")
if not os.path.exists(final_path):
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        final_path = zips[-1]
    else:
        raise FileNotFoundError("No model found to export")

export_model = PPO.load(final_path, device="cpu")
wrapper = PolicyWrapper(export_model.policy).cpu()
wrapper.eval()

dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)
with torch.no_grad():
    test_out = wrapper(dummy)
    print(f"Test output shape: {test_out.shape} (expected: [1, {NUM_ACTIONS}])")
    print(f"Test logits: {test_out[0, :5].numpy()} ... (first 5 of {NUM_ACTIONS})")

onnx_path = os.path.join(DRIVE_DIR, f"{VERSION}_jockey.onnx")

torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"],
    output_names=["actions"],
    dynamic_axes={"obs": {0: "batch"}, "actions": {0: "batch"}},
    opset_version=17,
)

# Verify
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(onnx_path)
test_obs = np.random.randn(4, OBS_SIZE).astype(np.float32)

with torch.no_grad():
    torch_out = wrapper(torch.from_numpy(test_obs)).numpy()
onnx_out = session.run(["actions"], {"obs": test_obs})[0]

max_diff = np.max(np.abs(torch_out - onnx_out))
print(f"\nExported to: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")
print(f"Max PyTorch vs ONNX diff: {max_diff:.2e}")
assert max_diff < 1e-5, f"Output mismatch: {max_diff}"
print("Validation passed!")
print(f"\nCopy this file to your TS app: apps/horse-racing/public/models/")

## Download ONNX

The exported `.onnx` file is already saved to Google Drive at `{DRIVE_DIR}/{VERSION}_jockey.onnx`. You can access it directly from your Drive or copy it to your TS app:

In [ ]:
onnx_path = os.path.join(DRIVE_DIR, f"{VERSION}_jockey.onnx")
if os.path.exists(onnx_path):
    size_kb = os.path.getsize(onnx_path) / 1024
    print(f"ONNX saved to Drive: {onnx_path} ({size_kb:.1f} KB)")
    print(f"\nTo use in the TS app, copy from Google Drive:")
    print(f"  Drive path: hr-checkpoints/{VERSION}/{VERSION}_jockey.onnx")
    print(f"  Dest: apps/horse-racing/public/models/{VERSION}_jockey.onnx")
else:
    print(f"ONNX file not found at {onnx_path} — run the export cell first.")